In [ ]:
import chipwhisperer as cw
import time
import numpy as np
from IPython.display import clear_output
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import struct
from utils import *
import bz2
import pandas as pd
import optuna
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ChipWhisperer setup

In [ ]:
scope = cw.scope(sn='INSERT_YOUR_SERIAL_NUMBER_HERE')  # Replace with your ChipWhisperer serial number
target = cw.target(scope, cw.targets.SimpleSerial2)

In [ ]:
scope.default_setup()

In [ ]:
%%bash
cd ../firmware/tiny-ml/
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1 -j

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/tiny-ml/tiny-ml-CWHUSKY.hex")

# Testing Wakeword and Faulting

In [ ]:
cwa = CW_Agent_tiny_ml(target)

## Sending Fault No and Command for specific data on the board 
wwd_index = 0x01 # some audio data on the board
unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
cwa.send_fault_no_and_cmd(0x00, 0x01, 0x05, wwd_index, unprotected_protected_cmd)

## Sending Fault No and Command for entire dataset
# wwd_index = 0x01 # some audio data on the board
# unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
# cwa.send_fault_no_and_cmd(0x00, 0x01, 0x00, wwd_index, unprotected_protected_cmd)

In [ ]:
wwd_data_path = base_dir + 'firmware/wakeWord-cmsisnn/dataset/audio_data.npz'
audio_Data = np.load(wwd_data_path)
X_data = audio_Data['features']
Y_data = audio_Data['labels']


In [ ]:
# only log critical messages
cw.set_all_log_levels(cw.logging.CRITICAL)

In [ ]:
scope.gain.db = 25

In [ ]:
scope.clock.adc_mul = 8

In [ ]:
scope.adc.samples = 13792
scope.adc.clear_clip_errors()

In [ ]:
# enable voltage glitching
scope.glitch.enabled = True # enable or disable faults
if scope.glitch.enabled:
    scope.glitch.clk_src = "pll" # use pll as source
    scope.io.glitch_hp = True # high-power glitch
    scope.io.glitch_lp = False # low-power glitch
    scope.glitch.output = "glitch_only" # just glitch, no combination with clock
    scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called based on trigger
    scope.adc.timeout = 0.5 # seconds to wait until capture is aborted
    scope.glitch.offset = 1500 # offset from rising clock edge to glitch rising edge, in # phase shift steps, at phase_shift_steps / 2 this is aligned with the falling edge of clock
    scope.glitch.width = 2100 # width of single glitch, # phase shift steps, max=phase_shift_steps / 2
    scope.glitch.num_glitches = 1 # number of glitches to generate
    scope.glitch.ext_offset = 0 # how long glitch waits between trigger and glitch, in clock cycles. If num_glitches > 1, should be list for each glitch
    scope.glitch.repeat = 1 # number of glitch pulses to generate per trigger, if num_glitches > 1, should be list for each glitch
    scope.gain.db = 18 # lower gain when glitching so that values are not clipped
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True

In [ ]:
## Sending Fault No and Command for specific data on the board 
wwd_index = 0x01 # some audio data on the board
unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
cwa.send_fault_no_and_cmd(0x00, 0x01, 0x05, wwd_index, unprotected_protected_cmd)
scope.glitch.enabled = False
for s in range(5):
    scope.arm()
    # trigger
    cwa.run_wake_word_without_return(unprotected_protected_cmd)
    test = scope.capture()
    scope.io.vglitch_reset()
    pred = target.simpleserial_read_witherrors(cmd='p', pay_len=1, timeout=0)['payload'][0]
    print(s, scope.adc.trig_count / scope.clock.adc_mul, scope.adc.trig_count)

scope.glitch.enabled = True

In [ ]:
## Sending Fault No and Command for specific data on the board 
wwd_index = 0x01 # some audio data on the board
unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
cwa.send_fault_no_and_cmd(0x00, 0x01, 0x05, wwd_index, unprotected_protected_cmd)
cwa.send_input_data_wwd(X_data[246], unprotected_protected_cmd)
scope.arm()
cwa.run_wake_word_without_return(unprotected_protected_cmd)
scope.capture()
scope.io.vglitch_reset()
pred = target.simpleserial_read_witherrors(cmd='p', pay_len=1, timeout=300)
valid = pred['valid']
if valid:
    response = pred['payload']
    raw_serial = pred['full_response']
    error_code = pred['rv']

print(pred)

In [ ]:
def insert_glitch(unprotected_protected_cmd, wwd_data, actual_label, do_glitch=True, is_data_set=False):
    """
    Reboot, optionally inject a glitch pulse, run ImageCNN on `image`,
    and ALWAYS capture Prediction + intermediates.
    If do_glitch=False → disable glitch engine and do one clean inference.
    If do_glitch=True  → enable glitch engine and fire one pulse.
    Returns a dict with:
      - ok:              bool, whether scope.capture() succeeded
      - serial_ok:       bool, whether we got ANY serial payload back
      - Prediction:      int
      - FC1_out:         list or array
      - FC2_out:         list or array
      - FC3_out:         list or array
      - Softmax_out:     list or array
      - DQ_out:          list or array
    """

    # pull out the intermediate arrays and force them to lists
    def tolist(x):
        return x.tolist() if hasattr(x, "tolist") else list(x)
        
    state_mcu_is_in = -5 # original placeholder
    fc1_output = []
    fc2_output = []
    fc3_output = []
    softmax_output =  []
    dequantize_output =  []
    
    try:
        actual_label = int(actual_label)
        unprotected_protected_cmd = int(unprotected_protected_cmd)
        
        # reboot & send image
        # cwa.reboot(scope)
        ## Sending Fault No and Command for specific label on the board 
        input_data = wwd_data # input data
        if is_data_set:
            wwd_index = actual_label # wakeword actual label
            cwa.send_fault_no_and_cmd(0x00, 0x01, 0x00, wwd_index, unprotected_protected_cmd)
            cwa.send_input_data_wwd(input_data, unprotected_protected_cmd)
        else:
            wwd_index = 0x01 # some audio data on the board
            cwa.send_fault_no_and_cmd(0x00, 0x01, 0x05, wwd_index, unprotected_protected_cmd)

        ok = None
        fired = None
        serial_ok = None
        pred = None
        
        if do_glitch:
            # glitch
            # scope.glitch.enabled = True
            # scope.glitch.repeat = 1
            scope.arm()
            cwa.run_wake_word_without_return(unprotected_protected_cmd)
    
            # record whether glitch “caught” or not
            ok = scope.capture()
            fired = not ok            # <— invert: False→caught pulse, True→timeout
            # print(f"[GLITCH] width={scope.glitch.width}, offset={scope.glitch.offset}, "
            #           f"ext_offset={scope.glitch.ext_offset} → capture_ok={fired}")
            scope.io.vglitch_reset()
        else:
            # no glitch: just run inference
            scope.glitch.enabled = False
            pred = cwa.predict_wake_word(input_data, unprotected_protected_cmd)
            state_mcu_is_in = -1
            fc1_output = tolist(cwa.get_fc1_output(unprotected_protected_cmd))
            fc2_output = tolist(cwa.get_fc2_output(unprotected_protected_cmd))
            fc3_output = tolist(cwa.get_fc3_output(unprotected_protected_cmd))
            softmax_output = tolist(cwa.get_softmax_output(unprotected_protected_cmd))
            dequantize_output = tolist(cwa.get_dequantize_output(unprotected_protected_cmd))
            ok = False
            fired = ok
            serial_ok = True
            print("[BASELINE] no glitch")

        if do_glitch:
            if ok:
                pred = None
                state_mcu_is_in = 0
                fc1_output = []
                fc2_output =  []
                fc3_output =  []
                softmax_output =  []
                dequantize_output =  []
            else:
                val = target.simpleserial_read_witherrors(cmd='p', pay_len=1, timeout=500)
                serial_ok = bool(val.get('valid', False))
                if serial_ok is False:
                    pred = None
                    state_mcu_is_in = 0
                    fc1_output = []
                    fc2_output =  []
                    fc3_output =  []
                    softmax_output =  []
                    dequantize_output =  []
                else:
                    pred = val['payload'][0]
            
                    if pred != actual_label:
                        state_mcu_is_in = 1
                        fc1_output = tolist(cwa.get_fc1_output(unprotected_protected_cmd))
                        fc2_output = tolist(cwa.get_fc2_output(unprotected_protected_cmd))
                        fc3_output = tolist(cwa.get_fc3_output(unprotected_protected_cmd))
                        softmax_output = tolist(cwa.get_softmax_output(unprotected_protected_cmd))
                        dequantize_output = tolist(cwa.get_dequantize_output(unprotected_protected_cmd))
                    else:
                        state_mcu_is_in = -1
                        fc1_output = tolist(cwa.get_fc1_output(unprotected_protected_cmd))
                        fc2_output = tolist(cwa.get_fc2_output(unprotected_protected_cmd))
                        fc3_output = tolist(cwa.get_fc3_output(unprotected_protected_cmd))
                        softmax_output = tolist(cwa.get_softmax_output(unprotected_protected_cmd))
                        dequantize_output = tolist(cwa.get_dequantize_output(unprotected_protected_cmd))

        # Reboot ONLY if hang (0) or fault (1)
        if state_mcu_is_in in (0, 1):
            ## reboot 
            cwa.reboot(scope)
            
        return {
            'fired':           bool(fired),
            'serial_ok':       serial_ok,
            'Prediction':      pred,
            'FC1_out':         fc1_output,
            'FC2_out':         fc2_output,
            'FC3_out' :        fc3_output,
            'Softmax_out':     softmax_output,
            'DQ_out':          dequantize_output,
            'state_mcu_is_in': state_mcu_is_in ## -1 = normal, 0 = reset, 1 = success 
        }

    except Exception as ex:
        print(f"[ERROR] {ex}")
        # Consider exceptions as a hang; reset once to recover.
        try:
            cwa.reboot(scope)
        except Exception as ex2:
            print(f"[ERROR during recovery reset] {ex2}")
            
        # if *any* step fails, return a dict with ok=False and None outputs
        return {
            'fired':           False,
            'serial_ok':       False,
            'Prediction':      None,
            'FC1_out':         None,
            'FC2_out':         None,
            'FC3_out' :        None,
            'Softmax_out':     None,
            'DQ_out':          None,
            'state_mcu_is_in': 0 ## -1 = normal, 0 = reset, 1 = success 
        }


# objective collects over images & repeats, and builds the 'details' table ---

def objective(trial: optuna.Trial):
    # set glitch params
    shift_steps = scope.glitch.phase_shift_steps
    scope.glitch.width      = trial.suggest_int("width",      0,  4000,  step=100)
    scope.glitch.offset     = trial.suggest_int("offset",     0,  4000,  step=100)
    scope.glitch.ext_offset = trial.suggest_int("ext_offset", 0, 78579, step=100)
    scope.glitch.repeat = trial.suggest_int("repeat", 1, 5, step=1)
    
    num_faults  = 0
    num_resets  = 0
    num_repeats = 50
    cmd         = 0x05 # 0x05 for unprotected; 0x06 for protected
    img = X_data[3] # replace later
    actual = 0x00     # replace later

    # include headers for clarity
    header = [
      "fired","serial_ok","Prediction","Actual","WWD_Data No",
      "FC1_out","FC2_out","FC3_out","Softmax_out","DQ_out",
      "Resets_so_far","Faults_so_far"
    ]
    details = [header]

    for i in range(1):
        for _ in range(num_repeats):
            out = insert_glitch(cmd, img, actual)

            ## 'state_mcu_is_in' -1 = normal, 0 = reset, 1 = success
            if out['state_mcu_is_in'] == 0:
                num_resets += 1
            elif out['state_mcu_is_in'] == 1:
                num_faults += 1

            # record its OK flag plus all fields
            details.append([
                out['fired'],
                out['serial_ok'],
                out['Prediction'],
                actual,
                i,
                out['FC1_out'],
                out['FC2_out'],
                out['FC3_out'],
                out['Softmax_out'],
                out['DQ_out'],
                num_resets,
                num_faults
            ])

    # expose both the scalar metric and the full table
    trial.set_user_attr("resets",     num_resets)
    trial.set_user_attr("details",    details)

    # Optuna will maximize this
    return float(num_faults)

In [ ]:
study_name = f"wake_word_glitchv_optuna_test_new_50_per_trial_updated"
study = optuna.create_study(
    storage=f"sqlite:///{optuna_dir}{study_name}.db", 
    sampler=optuna.samplers.TPESampler(),
    study_name=study_name, 
    direction='maximize',
    load_if_exists=True,
)

In [ ]:
scope.glitch.enabled = True
study.optimize(objective, n_trials=2_500, show_progress_bar=True)

In [ ]:
success_trials = [t for t in study.trials if t.state == TrialState.COMPLETE and t.values[0] > 0]
print(len(success_trials))

In [ ]:
# after you’ve run:
#   study.optimize(objective, n_trials=…)
best = study.best_trial

# pull out the raw details table
details = best.user_attrs["details"]

# option A: simple Python list
# skip header row
preds = [row[2] for row in details[1:]]
print("Predictions for best trial:", preds)

# # # first row is header
header = details[0]
df = pd.DataFrame(details[1:], columns=header)


In [ ]:
df

In [ ]:
print("Best trial:")
print(study.best_trial.params, "→ error", study.best_value)

In [ ]:
import optuna
import pandas as pd
from optuna.trial import TrialState
from optuna.study import StudyDirection

# --- gather completed trials ---
completed = study.get_trials(deepcopy=False, states=(TrialState.COMPLETE,))
completed = [t for t in completed if t.value is not None]  # guard against None

print(f"Completed trials: {len(completed)}")

# If you want to keep your "success" notion (e.g., value > 0), do:
success_trials = [t for t in completed if t.value > 0]
print(f"Successful trials (>0): {len(success_trials)}")

# --- show top-k trials ---
top_k = 200  # change to whatever you want
reverse = (study.direction == StudyDirection.MAXIMIZE)  # True → sort high→low
top_trials = sorted(completed, key=lambda t: t.value, reverse=reverse)[:top_k]

print(f"\nTop {len(top_trials)} trials:")
for rank, t in enumerate(top_trials, 1):
    print(f"#{rank} | trial={t.number:4d} | value={t.value:.6f} | params={t.params}")

# --- nice tabular view (params flattened) ---
top_rows = []
for rank, t in enumerate(top_trials, 1):
    row = {"rank": rank, "trial_number": t.number, "value": t.value}
    row.update(t.params)
    top_rows.append(row)

top_df = pd.DataFrame(top_rows).sort_values("rank", ignore_index=True)
print("\nTop trials dataframe:")
print(top_df)

# --- your existing 'best trial' details/predictions block (kept, with minor safety checks) ---
best = study.best_trial
details = best.user_attrs.get("details")

if details:
    # first row is header
    header = details[0]
    df_details = pd.DataFrame(details[1:], columns=header)

    # if your 3rd column is predictions, keep your original extraction
    preds = [row[2] for row in details[1:]]
    print("\nPredictions for best trial:", preds)
else:
    print("\nNo 'details' found in best_trial.user_attrs.")

# --- OPTIONAL: full trials dataframe for quick ad-hoc filtering/sorting ---
# df_all = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
# df_all = df_all[df_all["state"] == "COMPLETE"].sort_values("value", ascending=not reverse)
# print(df_all.head(top_k))


## Brute force search

In [ ]:
import csv

def brute_force_glitch_search(
    cmd,               # 0x05=unprotected; 0x06=protected
    wwd_data,          # data     
    actual_no,         # actual no / label
    input_data_no,     # input data number 
    num_repeats=50,    # repeat 50 times per input image
    widths=None,       # list of widths to try
    offsets=None,      # list of offsets to try
    ext_offsets=None,  # list of ext_offsets to try
    repeats=None,      # number of glitch pulses to generate per trigger
    do_glitch = True,  # do glitch or not
    is_data_set=False  # using dataset or not
):
    # ## auto-wrap scalars into singleton lists
    # if not hasattr(input_image, "__iter__") or isinstance(input_image, (bytes, str)):
    #     input_image = [input_image]
    # if not hasattr(actual_no, "__iter__") or isinstance(actual_no, (bytes, str)):
    #     actual_no = [actual_no]
        
    ## default grids
    shift_steps = scope.glitch.phase_shift_steps
    # if widths      is None: widths      = list(range(0, shift_steps, 1000))
    # if offsets     is None: offsets     = list(range(0, shift_steps, 1000))
    # if ext_offsets is None: ext_offsets = list(range(0, int(0.05*105565), 1200))
    # if repeats     is None: repeats     = list(range(1, 11))
        
    # if widths      is None: widths      = [2000]
    # if offsets     is None: offsets     = [1000] #2k, 3k, 0
    # if ext_offsets is None: ext_offsets = [0] #3600 #4800
    
    if widths      is None: widths      = [3300] # fix
    if offsets     is None: offsets     = [3900] # fix 
    # if ext_offsets is None: ext_offsets = [int(x/100*105565 )for x in range(0,105, 5)]  ## similar to start time
    # if ext_offsets is None: ext_offsets = list(range(0, 78579, 100))
    if ext_offsets is None: ext_offsets = list(range(15700, 26000, 100))
    # if ext_offsets is None: ext_offsets = [24200]
    if repeats     is None: repeats     = [4] ## num glitches fix 

    best_faults  = -1
    best_params  = {}
    full_details = []  

    header = [
      "width", "offset", "ext_offset", "Num_Glitches",
      "Correct", "Faulty", "Fail",
      "Prediction", "Actual", "Input_Data No",
      "FC1_out","FC2_out","FC3_out","Softmax_out","DQ_out"
    ]
    full_details.append(header)

    total = len(widths) * len(offsets) * len(ext_offsets) * len(repeats)
    pbar = tqdm(total=total, desc="Grid search")
    
    # Brute‐force loops
    for w in widths:
        scope.glitch.width = w
        for o in offsets:
            scope.glitch.offset = o
            for eo in ext_offsets:
                scope.glitch.ext_offset = eo
                for num_glitches in repeats:
                    scope.glitch.repeat = num_glitches

                    num_faults  = 0
                    num_resets  = 0
                    num_correct = 0

                    # 3) Run through the image
                    for t in (range(num_repeats)):
                        out = insert_glitch(cmd, wwd_data, actual_no, do_glitch=do_glitch, is_data_set=is_data_set)

                        ## 'state_mcu_is_in' -1 = normal, 0 = reset, 1 = success
                        if out['state_mcu_is_in'] == 0:
                            num_resets += 1
                        elif out['state_mcu_is_in'] == 1:
                            num_faults += 1
                        elif out['state_mcu_is_in'] == -1:
                            num_correct += 1

                        if out['Prediction'] is None:
                            out['Prediction'] = []
                        
                        # record each trial’s outcome
                        full_details.append([
                            w, o, eo, num_glitches,
                            num_correct, num_faults, num_resets,
                            out['Prediction'], actual_no, input_data_no, 
                            out['FC1_out'],
                            out['FC2_out'],
                            out['FC3_out'],
                            out['Softmax_out'],
                            out['DQ_out']
                        ])
                    
                    # 4) Track best
                    if num_faults > best_faults:
                        best_faults = num_faults
                        best_params = {
                            "width": w,
                            "offset": o,
                            "ext_offset": eo,
                            "Num_Glitches": num_glitches,
                            "faults": num_faults,
                            "resets": num_resets
                        }
                        
                    pbar.update(1)

    pbar.close()
    
    # Return the winner plus full log
    return best_params, full_details

# cmd = 0x05     # 0x05=unprotected; 0x06=protected
# img = X_data[246]# replace later
# # actual = 0x01     # replace later
# actual = Y_data[246]   # replace later
# input_data_no = 0
# scope.glitch.enabled = True
# best, log = brute_force_glitch_search(cmd, img, actual, input_data_no, do_glitch=True, is_data_set=False)

# print("Best parameters →", best)

# filename = "fine_grain_search/wake_word/wake_word_unprotected_fine_grain_new.csv"

# with open(filename, "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerows(log)

## Testing Unprotected No Fault

In [ ]:
## Sending Fault No and Command for specific data on the board 
# wwd_index = 0x01 # some audio data on the board
# unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
# cwa.send_fault_no_and_cmd(0x00, 0x01, 0x05, wwd_index, unprotected_protected_cmd)

## Sending Fault No and Command for entire dataset
wwd_index = 0x01 # some audio data on the board
unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
cwa.send_fault_no_and_cmd(0x00, 0x01, 0x00, wwd_index, unprotected_protected_cmd)

header = ['Detection', 'Actual', 'Input No', 'FC1_out', 'FC2_out', 'FC3_out', 'Soft_out', 'DQ_Out']
predicted_labels = []
write = []

for i in tqdm(range(len(X_data))): ## change to len of 0 images
    for j in range (1):  
        result_tmp = {"Detection":[], 'FC1':[], 'FC2':[], 'FC3':[], 'Softmax': [], 'DQ_Out': []}
        pred = cwa.predict_wake_word(X_data[i], unprotected_protected_cmd)
        result_tmp['Detection'].append(pred)
        fc1_output = cwa.get_fc1_output(unprotected_protected_cmd)
        fc2_output = cwa.get_fc2_output(unprotected_protected_cmd)
        fc3_output = cwa.get_fc3_output(unprotected_protected_cmd)
        softmax_output = cwa.get_softmax_output(unprotected_protected_cmd)
        dequantize_output = cwa.get_dequantize_output(unprotected_protected_cmd)
        result_tmp['FC1'] = fc1_output
        result_tmp['FC2'] = fc3_output
        result_tmp['FC3'] = fc3_output
        result_tmp['Softmax'] = softmax_output
        result_tmp['DQ_Out'] = dequantize_output
    write.append([result_tmp['Detection'], Y_data[i], i, result_tmp['FC1'], result_tmp['FC2'], result_tmp['FC3'], result_tmp['Softmax'], result_tmp['DQ_Out']])

final = pd.DataFrame(write, columns=header)
filename = data_dir + 'wake_word_run_unprotected_no_fault.csv'
final.to_csv(filename, encoding='utf-8', index=False)

In [ ]:
final

## Testing Protected No Fault

In [ ]:
## Sending Fault No and Command for specific data on the board 
# wwd_index = 0x01 # some audio data on the board
# unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected
# cwa.send_fault_no_and_cmd(0x00, 0x01, 0x05, wwd_index, unprotected_protected_cmd)

## Sending Fault No and Command for entire dataset
wwd_index = 0x01 # some audio data on the board
unprotected_protected_cmd = 0x06 # 0x05 for unprotected; 0x06 for protected
cwa.send_fault_no_and_cmd(0x00, 0x01, 0x00, wwd_index, unprotected_protected_cmd)

header = ['Detection', 'Actual', 'Input No', 'FC1_out', 'FC2_out', 'FC3_out', 'Soft_out', 'DQ_Out']
predicted_labels = []
write = []

for i in tqdm(range(len(X_data))): ## change to len of 0 images
    for j in range (1):  ## change to 50
        result_tmp = {"Detection":[], 'FC1':[], 'FC2':[], 'FC3':[], 'Softmax': [], 'DQ_Out': []}
        pred = cwa.predict_wake_word(X_data[i], unprotected_protected_cmd)
        result_tmp['Detection'].append(pred)
        fc1_output = cwa.get_fc1_output(unprotected_protected_cmd)
        fc2_output = cwa.get_fc2_output(unprotected_protected_cmd)
        fc3_output = cwa.get_fc3_output(unprotected_protected_cmd)
        softmax_output = cwa.get_softmax_output(unprotected_protected_cmd)
        dequantize_output = cwa.get_dequantize_output(unprotected_protected_cmd)
        result_tmp['FC1'] = fc1_output
        result_tmp['FC2'] = fc3_output
        result_tmp['FC3'] = fc3_output
        result_tmp['Softmax'] = softmax_output
        result_tmp['DQ_Out'] = dequantize_output
    write.append([result_tmp['Detection'], Y_data[i], i, result_tmp['FC1'], result_tmp['FC2'], result_tmp['FC3'], result_tmp['Softmax'], result_tmp['DQ_Out']])

final = pd.DataFrame(write, columns=header)
filename = data_dir + 'wake_word_run_protected_no_fault.csv'
final.to_csv(filename, encoding='utf-8', index=False)

In [ ]:
final

## Faulting Unprotected

In [ ]:
import csv
## Sending Fault No and Command for entire dataset
unprotected_protected_cmd = 0x05 # 0x05 for unprotected; 0x06 for protected

filename = "confusion_matrices/wake_word/data/wake_word_unprotected"
# ext_offsets = [31900] 
# ext_offsets = [0.1*78579, 0.15*78579, 0.2*78579] 
# ext_offsets = [0.01*x*78579 for x in range(0, 105, 5)] 
# ext_offsets = [0.01*x*78579 for x in range(20, 31, 1)] 
ext_offsets = [29200] 
repeats = [1]

for ext_offset in tqdm(ext_offsets):
    start_time = ext_offset/78579
    with open(filename + f"_{start_time:.3f}_start_time.csv", "w", newline="") as f:
        writer = csv.writer(f)
        header_written = False
        
        for i in tqdm(range(len(X_data))): ## change to len of 0 images    
            _, log = brute_force_glitch_search(unprotected_protected_cmd, X_data[i], Y_data[i], i, num_repeats=1, ext_offsets=[ext_offset], repeats=[repeats], do_glitch=True, is_data_set=True)
    
            if not header_written:
                # write everything (incl. header row at log[0])
                writer.writerows(log)
                header_written = True
            else:
                # skip the log[0] header row
                writer.writerows(log[1:])

## Faulting Protected

In [ ]:
import csv
## Sending Fault No and Command for entire dataset
unprotected_protected_cmd = 0x06 # 0x05 for unprotected; 0x06 for protected

filename = "confusion_matrices/wake_word/data/wake_word_unprotected"
ext_offsets = [55000] ## change inject time here to the percentage of total time
# ext_offsets = [0.1*78579, 0.15*78579, 0.2*78579]  ## change total time here
# ext_offsets = [0.01*x*78579 for x in range(0, 105, 5)]  ## change total time here
# ext_offsets = [0.01*x*78579 for x in range(20, 31, 1)]  ## change total time here
repeats = [4]

for ext_offset in tqdm(ext_offsets):
    start_time = ext_offset/78579
    with open(filename + f"_{start_time:.2f}_start_time.csv", "w", newline="") as f:
        writer = csv.writer(f)
        header_written = False
        
        for i in tqdm(range(len(usps_data_new))): ## change to len of 0 images    
            _, log = brute_force_glitch_search(unprotected_protected_cmd, X_data[i], Y_data[i], i, num_repeats=1, ext_offsets=[ext_offset], repeats=[repeats], do_glitch=True, is_data_set=True)
    
            if not header_written:
                # write everything (incl. header row at log[0])
                writer.writerows(log)
                header_written = True
            else:
                # skip the log[0] header row
                writer.writerows(log[1:])

## Confusion Matrices

In [ ]:
import numpy as np
import pandas as pd
import ast
import os
import seaborn as sns
import matplotlib.pyplot as plt

def parse_predictions(val):
    # Already a Python list
    if isinstance(val, list):
        return val
    # Missing / empty -> no prediction
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    # If it's a plain int (or numpy int), wrap it
    if isinstance(val, (int, np.integer)):
        return [int(val)]
    # Strings like "7", "[]", "[1,2]" -> eval safely
    if isinstance(val, str):
        s = val.strip()
        if s == "" or s == "[]":
            return []
        try:
            obj = ast.literal_eval(s)
        except Exception:
            # Fallback: maybe it's "7"
            try:
                return [int(s)]
            except Exception:
                return []
        # Normalize to list
        if isinstance(obj, list):
            return obj
        if isinstance(obj, (int, np.integer)):
            return [int(obj)]
        if isinstance(obj, tuple):
            return list(obj)
        return []
    # Anything else -> no prediction
    return []

def print_confusion_matrix(data_frame):
    # Define the labels for actual and predicted classes
    actual_labels = list(range(2))
    predicted_labels = list(range(2)) + ["Others", "Reset"]

    # Initialize an empty confusion matrix with zeros
    CM_df = pd.DataFrame(0, index=actual_labels, columns=predicted_labels)

    # Populate the confusion matrix by iterating over the data
    for _, row in data_frame.iterrows():
        actual = int(row["Actual"])
        predictions = parse_predictions(row["Prediction"])

        # If predictions list is empty, count as "Reset"
        if not predictions:
            CM_df.at[actual, "Reset"] += 1
        else:
            # Check if each predicted class is within the defined range, else mark as "Others"
            for pred in predictions:
                if pred in CM_df.columns:
                    CM_df.at[actual, pred] += 1
                else:
                    CM_df.at[actual, "Others"] += 1
    return CM_df

def normalize_confusion_matrix(CM_df):
    """
    Normalize the confusion matrix rows to percentages.
    """
    CM_df.drop(["Others", "Reset"], axis=1, inplace=True)
    CM_normalized = CM_df.div(CM_df.sum(axis=1), axis=0) * 100  # Convert to percentages
    CM_normalized = CM_normalized.fillna(0)  # Replace NaN with 0 for rows with no samples
    return CM_normalized

def visualize_confusion_matrix(CM_df):
    plt.figure(figsize=(10, 8))  # Adjust the figure size as needed
    sns.heatmap(CM_df, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
    plt.title("Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

def visualize_confusion_matrix_with_accuracy(CM_df, accuracy, save_path, title="Confusion Matrix"):
    """
    Visualize the normalized confusion matrix with accuracy displayed in the title.
    """
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(CM_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar=False, annot_kws={"size": 10})
    plt.title(f"{title}\nAccuracy: {accuracy:.2%}", fontsize=14)
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.savefig(save_path)
    plt.show()

def print_accuracy(confusion_matrix):
    # Calculate true positives (diagonal elements)
    true_positives = sum(confusion_matrix.iloc[i, i] for i in range(2))  # Only for classes 0-9
    
    # Exclude "Others" and "Reset" from total samples
    total_samples = confusion_matrix.iloc[:, :2].values.sum()  # Sum only columns 0-9

    # Compute accuracy
    accuracy = true_positives / total_samples if total_samples > 0 else 0

    print("Accuracy (excluding Others and Resets):%.2f%%" % (accuracy * 100))
    return accuracy


start_time = [0.01*x for x in range(0, 105, 5)]
# start_time = [0.01*x for x in range(20, 31, 1)]
start_time = [29200/78579]
start_time = [31700/78579, 31800/78579, 32000/78579, 32100/78579]
start_time = [31900/78579, 30400/78579, 28600/78579, 28800/78579, 29200/78579]
filename = "confusion_matrices/wake_word/data/wake_word_unprotected"
for time in start_time:
    faulty_data_unprotected = pd.read_csv(filename + f"_{time:.3f}_start_time.csv")
    print(f"Confusion Matrix (No Protection):")
    confusion_matrix = print_confusion_matrix(faulty_data_unprotected)
    confusion_matrix_normalized = normalize_confusion_matrix(confusion_matrix)
    accuracy = print_accuracy(confusion_matrix)
    fig_name = f"confusion_matrices/wake_word/plots/Wake_word_Confusion_matrix_unprotected_fault_{time:.3f}_start_time.pdf"
    # time = 24200/105565
    print(f"{time:.2f}")  
    visualize_confusion_matrix_with_accuracy(confusion_matrix_normalized, accuracy, fig_name, title=f"Confusion Matrix (Fault at {time:.3f}t)")


faulty_data_protected= pd.read_csv('wake_word_protected_0.20_start_time.csv')
print(f"Confusion Matrix (Protection):")
confusion_matrix = print_confusion_matrix(faulty_data_protected)
confusion_matrix_normalized = normalize_confusion_matrix(confusion_matrix)
accuracy = print_accuracy(confusion_matrix)
fig_name = 'Wake_word_Confusion_matrix_protected_fault.pdf'
time = 24200/105565
print(f"{time:.1f}")  
visualize_confusion_matrix_with_accuracy(confusion_matrix_normalized, accuracy, fig_name, title=f"Confusion Matrix (Fault at {time:.1f}t)")


In [ ]:
target.dis()
scope.dis()